In [0]:
%run ../init_framework_bronze

In [0]:
# Get Batch_Id
batch_id = get_batch_id()

In [0]:
from pyspark.sql.functions import lit

# 1. LANDING -> BRONZE
# autoloader handles the file discovery and schema enforcement from landing
df_raw = read_autoloader(spark, SRC_LOANS, loans_schema).withColumn("_batch_id", lit(batch_id))

# 2. AUDIT METADATA
# Add Standardized Bronze Audit Metadata
df_final = add_bronze_metadata(df_raw)

# 3. STREAM SINK
# standard append to bronze delta table. 
# checkpoint handles file tracking in the landing zone.
query = write_append_stream(df_final, LOANS_BRONZE, BRONZE_CHECKPOINT_LOANS)

# block notebook termination until micro-batch is committed
query.awaitTermination()

In [0]:
%sql
-- select count(1) from lending_risk.bronze.loans;
-- select * from lending_risk.bronze.loans limit 3;